# 케라스로 심층 합성곱 신경망 만들기

이 노트북에서 [LeNet-5](http://yann.lecun.com/exdb/publis/pdf/lecun-01a.pdf)과 비슷한 MNIST 손글씨 숫자를 분류하는 심층 합성곱 신경망을 만듭니다.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rickiepark/dl-illustrated/blob/master/notebooks/10-1.lenet_in_keras.ipynb)

#### 라이브러를 적재합니다.

In [3]:
from tensorflow import keras
from tensorflow.keras.datasets import mnist
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout

from tensorflow.keras.layers import Conv2D, MaxPooling2D 
# new! 합성곱 층과 최대 풀링 층을 구현하기 위해 ... 풀링은 활성화 맵을 줄이기 때문에 서브샘플링이라고도 부른다
# 점점 더 저렴한 계산 비용 때문에 풀링 층을 덜 자주 사용하는 것이 딥러닝에서 일반적인 경향이다

from tensorflow.keras.layers import Flatten 
# new! 다차원 배열을 1차원으로 만든다

2026-01-30 12:20:40.240839: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


#### 데이터를 적재합니다.

In [4]:
(X_train, y_train), (X_valid, y_valid) = mnist.load_data()

#### 데이터를 전처리합니다.

In [ ]:
# 2차원 이미지 구조를 28X28 그대로 유지. 흑백이라서 채널이 한개. 
# astype 으로 정수 데이터를 0에서 1사이 범위로 조정 
X_train = X_train.reshape(60000, 28, 28, 1).astype('float32')
X_valid = X_valid.reshape(10000, 28, 28, 1).astype('float32')

In [6]:
X_train /= 255
X_valid /= 255

In [7]:
n_classes = 10
y_train = keras.utils.to_categorical(y_train, n_classes)
y_valid = keras.utils.to_categorical(y_valid, n_classes)

print ('y_train :: ', y_train[0])
print ('y_valid :: ', y_valid[0])


y_train ::  [0. 0. 0. 0. 0. 1. 0. 0. 0. 0.]
y_valid ::  [0. 0. 0. 0. 0. 0. 0. 1. 0. 0.]


#### 데이터 적재와 전처리를 마치고 LeNet 신경망 모델을 만듭니다.

In [8]:
model = Sequential()

# 첫번째 합성곱 층 ==> 은닉층으로 밀집층이 아니라 합성곱층을 사용, 활성화 함수는 relu, strides 로 1을 적용
model.add(Conv2D(32, kernel_size=(3, 3), activation='relu', input_shape=(28, 28, 1)))

# 두번째 합성곱 층 - 풀링, 드롭아웃
model.add(Conv2D(64, kernel_size=(3, 3), activation='relu'))

# 풀링, 드롭아웃은 가중치가 없어서 독립적인 은닉층으로 세지 않는다
model.add(MaxPooling2D(pool_size=(2, 2)))   # 계산 복잡도를 낮춘다
model.add(Dropout(0.25))                    # 훈련 데이터에 대한 과대적합을 줄여준다

model.add(Flatten())        # Conv2D 가 만든 3차원 활성화 맵을 1차원 배열로 반환

# 밀집 은닉층, 드롭아웃 ==> 여기는 밀집 은닉층
model.add(Dense(128, activation='relu'))
model.add(Dropout(0.5))

# 출력층
model.add(Dense(n_classes, activation='softmax'))

/Users/ijaegu/.local/lib/python3.9/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [ ]:
# 합성곱 신경망 모델의 구조를 보여준다 ~~~~~
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 26, 26, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 24, 24, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 12, 12, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 12, 12, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 9216)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │     1,179,776 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,199,882 (4.58 MB)

 Trainable params: 1,199,882 (4.58 MB)

 Non-trainable params: 0 (0.00 B)

#### 모델을 설정합니다.

In [10]:
model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

#### 훈련!

In [12]:
model.fit(X_train, y_train, batch_size=128, epochs=10, verbose=1, validation_data=(X_valid, y_valid))

Epoch 1/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 179s 380ms/step - accuracy: 0.9727 - loss: 0.0882 - val_accuracy: 0.9873 - val_loss: 0.0366
Epoch 2/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 191s 408ms/step - accuracy: 0.9794 - loss: 0.0690 - val_accuracy: 0.9895 - val_loss: 0.0331
Epoch 3/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 185s 395ms/step - accuracy: 0.9825 - loss: 0.0565 - val_accuracy: 0.9904 - val_loss: 0.0283
Epoch 4/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 179s 381ms/step - accuracy: 0.9858 - loss: 0.0458 - val_accuracy: 0.9913 - val_loss: 0.0278
Epoch 5/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 176s 374ms/step - accuracy: 0.9872 - loss: 0.0402 - val_accuracy: 0.9906 - val_loss: 0.0295
Epoch 6/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 178s 380ms/step - accuracy: 0.9895 - loss: 0.0340 - val_accuracy: 0.9921 - val_loss: 0.0253
Epoch 7/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 179s 381ms/step - accuracy: 0.9897 - loss: 0.0309 - val_accuracy: 0.9926 - val_loss: 0.0270
Epoch 8/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 176s 374ms/step - accuracy: 0.9909 -